In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Agent Platform での Gemini グラウンディング入門 (v2)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Colab で開く
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fkazunori279%2Fgcp-blogs%2Fmain%2F20260602_gemini_grounding%2Fintro-grounding-gemini-v2-ja.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Colab Enterprise で開く
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/kazunori279/gcp-blogs/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Workbench で開く
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> GitHub で表示
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>共有:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| 著者 |
| --- |
| [Kazunori Sato](https://github.com/kazunori279) |

## 概要

[Agent Platform のグラウンディング](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/ground-gemini)を使用すると、生成テキストモデルで独自のドキュメントやデータに基づいたコンテンツを生成できます。この機能により、モデルは実行時にトレーニングデータを超えた情報にアクセスできます。モデルの応答を Google 検索結果や独自データでグラウンディングすることで、より正確で最新かつ関連性の高い応答を生成できます。

このノートブックでは、[Concierge AI](../README.md) デモアプリで使用されている **6 つのグラウンディング手法**を紹介します。Concierge AI は Gemini Live API を使った音声エージェントで、高級ホテルのコンシェルジュとして、Google 検索、Google マップ、Vector Search 2.0 のグラウンディングを活用します。

| # | グラウンディングの種類 | API / 機能 | Concierge ツール |
|---|---|---|---|
| 1 | Google 検索 | 組み込み `GoogleSearch` ツール | `web_search`（テキスト） |
| 2 | Google マップ | `GoogleMaps` ツール（場所検索、ルート、ルート沿い検索） | `web_search`（マップ）、`maps_route`、`maps_search_along_route` |
| 3 | **Vector Search 2.0** | `vectorsearch_v1beta` SDK（自動エンベディングによるセマンティック検索） | `vs2_search` |
| 4 | 画像検索 | `enhancedContent.imageSearch`（REST） | `web_search`（画像） |
| 5 | 動画検索 | 動画寄りクエリによる Google 検索 | `web_search`（動画） |
| 6 | 画像生成のための画像検索 | `searchTypes.imageSearch`（REST） | `request_postcard` → `illustrate_place` |

### 目標

このチュートリアルでは、以下の方法を学びます：

- Google 検索結果に基づいた LLM テキストの生成
- Google マップデータ（場所検索、ルート、ルート沿い検索）に基づいた LLM テキストの生成
- 自動エンベディング付き Vector Search 2.0 コレクションの作成とセマンティック検索の実行
- `enhancedContent.imageSearch` による画像検索
- Google 検索グラウンディングによる YouTube 動画の検索
- `searchTypes.imageSearch` による Google 画像検索結果に基づいた画像生成

このチュートリアルでは、以下の Google Cloud AI サービスとリソースを使用します：

- Agent Platform（Gemini API）
- Agent Platform Vector Search 2.0

実行する手順：

- 各種サンプル用の LLM とプロンプトの設定
- Google 検索および Google マップのグラウンディング付きプロンプトの送信
- ホテルナレッジベース用の自動エンベディング付き Vector Search 2.0 コレクションのセットアップ
- REST API による画像・動画の検索
- Google 画像検索結果に基づいた画像の生成

## 始める前に

### Google Cloud プロジェクトの設定

**以下の手順は、ノートブックの環境に関係なく必要です。**

1. [Google Cloud プロジェクトを選択または作成](https://console.cloud.google.com/cloud-resource-manager)します。初回アカウント作成時には、コンピューティング/ストレージ費用に使える $300 の無料クレジットが付与されます。
1. [プロジェクトの課金が有効になっていること](https://cloud.google.com/billing/docs/how-to/modify-project)を確認します。
1. [Agent Platform、Vector Search API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com,vectorsearch.googleapis.com) を有効にします。
1. このノートブックをローカルで実行する場合は、[Cloud SDK](https://cloud.google.com/sdk) のインストールが必要です。

### Google Gen AI SDK と Vector Search SDK for Python のインストール

このノートブックの実行に必要なパッケージをインストールします。

In [ ]:
# noqa: E225
%pip install --upgrade --quiet \
    google-genai \
    google-cloud-vectorsearch \
    google-auth \
    requests

### Google Cloud アカウントの認証

このノートブックを Google Colab で実行している場合は、環境の認証が必要です。以下のセルを実行してください。Agent Platform Workbench を使用している場合、この手順は不要です。

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

### Google Cloud プロジェクト情報の設定とクライアントの作成

Agent Platform を使い始めるには、既存の Google Cloud プロジェクトが必要で、[Agent Platform API を有効にする](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com)必要があります。

[プロジェクトと開発環境の設定](https://cloud.google.com/vertex-ai/docs/start/cloud-environment)について詳しくはこちらをご覧ください。

**プロジェクト ID がわからない場合**は、以下をお試しください：
* `gcloud config list` を実行
* `gcloud projects list` を実行
* サポートページ：[プロジェクト ID の確認](https://support.google.com/googleapi/answer/7014113)

Agent Platform で使用する `LOCATION` 変数を変更することもできます。[Agent Platform のリージョン](https://cloud.google.com/vertex-ai/docs/general/locations)について詳しくはこちらをご覧ください。

In [ ]:
import os  # noqa: E402

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

from google import genai  # noqa: E402

client = genai.Client(
    vertexai=True, project=PROJECT_ID, location=LOCATION
)

### ライブラリのインポート

In [ ]:
import base64
import json
import re

import google.auth
import google.auth.transport.requests as auth_requests
import requests as http_requests
from IPython.core.display import Markdown
from IPython.display import Image as IPImage
from IPython.display import display
from google.genai.types import (
    GenerateContentConfig,
    GenerateContentResponse,
    GoogleMaps,
    GoogleSearch,
    LatLng,
    Part,
    Tool,
    ToolConfig,
    RetrievalConfig,
)

### ヘルパー関数

In [ ]:
def print_grounding_data(response: GenerateContentResponse):
    """Print response with grounding citations in Markdown."""
    candidate = (
        response.candidates[0] if response.candidates else None
    )
    metadata = getattr(candidate, "grounding_metadata", None)

    if not metadata:
        print("Response does not contain grounding metadata.")
        display(Markdown(response.text))
        return

    ENCODING = "utf-8"
    text_bytes = response.text.encode(ENCODING)
    parts = []
    last = 0

    for support in metadata.grounding_supports or []:
        end = support.segment.end_index
        parts.append(text_bytes[last:end].decode(ENCODING))
        indices = support.grounding_chunk_indices
        parts.append(
            " " + "".join(f"[{i + 1}]" for i in indices)
        )
        last = end

    parts.append(text_bytes[last:].decode(ENCODING))
    parts.append("\n\n----\n## Grounding Sources\n")

    if chunks := metadata.grounding_chunks:
        parts.append("### Grounding Chunks\n")
        for i, chunk in enumerate(chunks, 1):
            ctx = (
                chunk.web
                or chunk.retrieved_context
                or chunk.maps
            )
            if not ctx:
                continue
            uri = (ctx.uri or "").replace(
                "gs://",
                "https://storage.googleapis.com/",
                1,
            ).replace(" ", "%20")
            title = ctx.title or "Source"
            parts.append(f"{i}. [{title}]({uri})\n")
            if getattr(ctx, "place_id", None):
                pid = ctx.place_id
                parts.append(
                    f"    - Place ID: `{pid}`\n\n"
                )
            if getattr(ctx, "text", None):
                parts.append(f"{ctx.text}\n\n")

    if metadata.web_search_queries:
        queries = metadata.web_search_queries
        parts.append(f"\n**Web Search Queries:** {queries}\n")
        if metadata.search_entry_point:
            sep = metadata.search_entry_point
            parts.append(
                f"\n**Search Entry Point:**\n"
                f"{sep.rendered_content}\n"
            )
    elif metadata.retrieval_queries:
        queries = metadata.retrieval_queries
        parts.append(
            f"\n**Retrieval Queries:** {queries}\n"
        )

    widget_attr = "google_maps_widget_context_token"
    if hasattr(metadata, widget_attr):
        token = getattr(metadata, widget_attr)
        if token:
            parts.append(
                f"\n**Maps Widget Token:**"
                f" `{token[:50]}...`\n"
            )

    display(Markdown("".join(parts)))


def get_authorized_session():
    """Create an authorized HTTP session for REST."""
    creds = google.auth.default()[0]
    return auth_requests.AuthorizedSession(creds)


def rest_endpoint(model: str) -> str:
    """Build the Agent Platform generateContent REST URL."""
    return (
        "https://aiplatform.googleapis.com/v1"
        f"/projects/{PROJECT_ID}"
        "/locations/global/publishers/google"
        f"/models/{model}:generateContent"
    )

Agent Platform から Gemini モデルを初期化します：

In [ ]:
MODEL_ID = "gemini-2.5-flash"  # @param {type: "string"}

---
## 1. Google 検索によるグラウンディング

Google 検索グラウンディングにより、Gemini はウェブ検索を実行し、その結果に基づいて回答を生成できます。Concierge AI アプリでは、`web_search` ツールがこの機能を使用して、最新のイベント情報、スケジュール、チケット価格などの質問に回答します。

### グラウンディングなし

In [ ]:
PROMPT = "What shows are playing in Las Vegas this week?"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
)

display(Markdown(response.text))

### Google 検索グラウンディングあり

`GoogleSearch` をツールとして追加することで、ソース引用付きのグラウンディングされた最新の回答を取得できます。

In [ ]:
google_search_tool = Tool(google_search=GoogleSearch())

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(tools=[google_search_tool]),
)

print_grounding_data(response)

---
## 2. Google マップによるグラウンディング

Google マップのグラウンディングにより、Gemini は場所の検索、経路案内、ルート沿いの POI 検索が可能になります。Concierge AI アプリでは、3 つのツールがこの機能を使用します：
- `web_search` — 周辺の場所を検索し、マップウィジェットを返す
- `maps_route` — 経路、距離、所要時間を取得
- `maps_search_along_route` — ドライブルート沿いの立ち寄りスポットを検索

### 場所検索

In [ ]:
google_maps_tool = Tool(google_maps=GoogleMaps(enable_widget=True))

PROMPT = "Find the best sushi restaurants near the Las Vegas Strip"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[google_maps_tool],
        tool_config=ToolConfig(
            retrieval_config=RetrievalConfig(
                lat_lng=LatLng(latitude=36.1699, longitude=-115.1398)
            ),
        ),
    ),
)

print_grounding_data(response)

### ルート — 経路案内、距離、所要時間

これは `maps_route` ツールで使用されるパターンです。Google マップのグラウンディングが有効な場合、モデルは自然言語のプロンプトからルーティングの意図を推測します。

In [ ]:
PROMPT = (
    "Give me driving directions from "
    "3950 S Las Vegas Blvd, Las Vegas, NV 89119 "
    "to Red Rock Canyon. "
    "Include the total travel time and distance."
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[Tool(google_maps=GoogleMaps(enable_widget=True))],
    ),
)

print_grounding_data(response)

### ルート沿い検索 — ドライブルート沿いの POI

これは `maps_search_along_route` ツールで使用されるパターンです。モデルがルートを計算し、ルート沿いの該当する POI を検索します。

In [ ]:
PROMPT = (
    "Find the top 3 coffee shops along the driving route "
    "from 3950 S Las Vegas Blvd, Las Vegas, NV 89119 to McCarran Airport."
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[Tool(google_maps=GoogleMaps(enable_widget=True))],
    ),
)

print_grounding_data(response)

---
## 3. Vector Search 2.0 によるグラウンディング

Vector Search 2.0 は、**自動エンベディング**機能により、独自データに対するセマンティック検索を提供します。エンベディングはベクトルスキーマの `vertex_embedding_config` を通じて `gemini-embedding-001` により自動生成されるため、エンベディングを自分で管理する必要はありません。

Concierge AI アプリでは、`vs2_search` ツールがホテルのナレッジベースコレクションにクエリを送信し、客室、ダイニング、スパ、アクセス、ポリシーなどに関するゲストの質問に回答します。

このセクションでは、小規模なデモコレクションを作成し、ホテル情報のアイテムを追加して、セマンティック検索クエリを実行します。

### Vector Search API の有効化

このセクションを使用する前に、プロジェクトで Vector Search API を有効にする必要があります。

In [ ]:
# noqa: E225
!gcloud services enable vectorsearch.googleapis.com --project={PROJECT_ID}

### 自動エンベディング付きコレクションの作成

In [ ]:
from google.cloud import vectorsearch_v1beta

VS2_LOCATION = "us-central1"
COLLECTION_ID = "notebook-hotel-demo"  # @param {type: "string"}
VS2_PARENT = f"projects/{PROJECT_ID}/locations/{VS2_LOCATION}"
COLLECTION_PATH = f"{VS2_PARENT}/collections/{COLLECTION_ID}"

vs_client = vectorsearch_v1beta.VectorSearchServiceClient()
data_client = vectorsearch_v1beta.DataObjectServiceClient()
search_client = vectorsearch_v1beta.DataObjectSearchServiceClient()

# Schema: data fields + auto-embedding configuration
DATA_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "content": {"type": "string"},
        "category": {"type": "string"},
    },
}

VECTOR_SCHEMA = {
    "content_embedding": {
        "dense_vector": {
            "dimensions": 768,
            "vertex_embedding_config": {
                "model_id": "gemini-embedding-001",
                "text_template": "{title} {category} {content}",
                "task_type": "RETRIEVAL_DOCUMENT",
            },
        },
    },
}

# Create the collection
try:
    vs_client.get_collection(
        request=vectorsearch_v1beta.GetCollectionRequest(name=COLLECTION_PATH)
    )
    print(f"Collection '{COLLECTION_ID}' already exists")
except Exception:
    print(f"Creating collection '{COLLECTION_ID}'...")
    op = vs_client.create_collection(
        request=vectorsearch_v1beta.CreateCollectionRequest(
            parent=VS2_PARENT,
            collection_id=COLLECTION_ID,
            collection={
                "data_schema": DATA_SCHEMA,
                "vector_schema": VECTOR_SCHEMA,
            },
        )
    )
    op.result()
    print(f"Collection '{COLLECTION_ID}' created")

### ホテルナレッジベースアイテムの追加

各アイテムには `title`、`content`、`category` があります。`vectors` フィールドは空のままにします。Vector Search 2.0 がスキーマで定義された `vertex_embedding_config` を使用して、エンベディングを自動生成します。

In [ ]:
HOTEL_INFO = [
    {
        "id": "room-01",
        "title": "Deluxe Room",
        "content": (
            "45 sqm room with king or twin beds, "
            "floor-to-ceiling windows overlooking the "
            "Imperial Palace gardens. Features include "
            "Simmons Beautyrest mattress, Egyptian cotton "
            "400-thread-count linens, 55-inch OLED TV, "
            "Nespresso machine, minibar, marble bathroom "
            "with rain shower and deep soaking tub, "
            "heated toilet seat, and complimentary Aesop "
            "amenities. Rate: from $450 per night."
        ),
        "category": "rooms",
    },
    {
        "id": "room-02",
        "title": "Premier Suite",
        "content": (
            "85 sqm suite with separate living area, "
            "dining table for 4, and panoramic Tokyo "
            "skyline views. Includes all Deluxe Room "
            "amenities plus Bang & Olufsen sound system, "
            "walk-in closet, dual vanity bathroom, and "
            "access to the Executive Lounge. Turndown "
            "service with Japanese wagashi sweets. "
            "Rate: from $1,000 per night."
        ),
        "category": "rooms",
    },
    {
        "id": "dine-01",
        "title": "Sakura Restaurant",
        "content": (
            "Our signature 2-Michelin-star kaiseki "
            "restaurant on the 36th floor, led by Chef "
            "Takeshi Yamamoto. Multi-course seasonal "
            "tasting menus featuring the finest "
            "Tsukiji-sourced seafood, Wagyu beef, and "
            "Kyoto vegetables. Omakase dinner: $250. "
            "Open for dinner 5:30pm-9:30pm (last "
            "seating). Reservations essential. "
            "Smart casual dress code."
        ),
        "category": "dining",
    },
    {
        "id": "dine-02",
        "title": "The Brasserie — All-Day Dining",
        "content": (
            "International restaurant on the lobby "
            "level serving breakfast buffet "
            "(6:30am-10:30am, $40), lunch a la carte "
            "(11:30am-2:30pm), and dinner "
            "(5:30pm-10pm). Highlights include the "
            "fresh sashimi station at breakfast, "
            "hand-rolled pasta, and Sunday champagne "
            "brunch ($85 with free-flow Veuve "
            "Clicquot). Children's menu available."
        ),
        "category": "dining",
    },
    {
        "id": "spa-01",
        "title": "Spa Overview and Treatments",
        "content": (
            "The Serenity Spa occupies the entire 5th "
            "floor with 12 treatment rooms, including "
            "3 couples' suites. Signature treatment: "
            "'Sakura Renewal' — 90-minute hot stone "
            "massage with cherry blossom oil ($195). "
            "Other popular treatments: Shiatsu massage "
            "(60 min, $125), Hinoki Facial (75 min, "
            "$155), and Matcha Body Scrub (45 min, "
            "$100). Open 9am-9pm daily."
        ),
        "category": "spa",
    },
    {
        "id": "pol-01",
        "title": "Check-in and Check-out Times",
        "content": (
            "Standard check-in: 3:00pm. Standard "
            "check-out: 12:00pm (noon). Early check-in "
            "from 1:00pm available for $50 (subject to "
            "availability). Late check-out until 3:00pm: "
            "$55; until 6:00pm: half-day rate. Luggage "
            "storage is complimentary before check-in "
            "and after check-out."
        ),
        "category": "policies",
    },
    {
        "id": "dir-01",
        "title": "Getting to the Hotel from Narita",
        "content": (
            "From Narita International Airport, take "
            "the Narita Express (N'EX) to Tokyo Station "
            "(approx. 60 minutes), then transfer to the "
            "Marunouchi Line to Otemachi Station "
            "(2 minutes). Exit C6b leads directly to "
            "the hotel lobby. Alternatively, our airport "
            "limousine bus departs every 30 minutes. "
            "Private car transfer: $250."
        ),
        "category": "directions",
    },
    {
        "id": "fit-01",
        "title": "Fitness Center",
        "content": (
            "24-hour fitness center on the 5th floor "
            "featuring Technogym equipment: 15 cardio "
            "machines (treadmills, bikes, ellipticals, "
            "rowing machines), full free weight section "
            "(dumbbells up to 50kg, squat rack, bench "
            "press), TRX suspension training, and "
            "stretching area with yoga mats. Personal "
            "training sessions available ($85/hour). "
            "Complimentary workout attire and towels."
        ),
        "category": "fitness",
    },
    {
        "id": "act-01",
        "title": "Cultural Experiences at the Hotel",
        "content": (
            "Daily cultural activities for guests "
            "(free of charge): Morning meditation in "
            "the zen garden (6:30am), origami workshop "
            "(10am, lobby lounge), Japanese calligraphy "
            "class (2pm, Culture Room), ikebana flower "
            "arrangement (4pm, Saturdays). Premium "
            "experiences: private tea ceremony with a "
            "tea master ($100/person, 60 min), sake "
            "tasting with sommelier ($85/person)."
        ),
        "category": "activities",
    },
    {
        "id": "wifi-01",
        "title": "WiFi and Internet Access",
        "content": (
            "Complimentary high-speed WiFi available "
            "throughout the hotel (lobby, rooms, "
            "restaurants, pool, spa). Network name: "
            "'GrandSapphire-Guest'. Connect and enter "
            "your room number and last name. Speed: "
            "200 Mbps download / 100 Mbps upload. "
            "Wired Ethernet connections available in "
            "all rooms. IT support: dial ext. 9 (24/7)."
        ),
        "category": "wifi",
    },
]

batch_request = [
    {
        "data_object_id": item["id"],
        "data_object": {
            "data": {
                "title": item["title"],
                "content": item["content"],
                "category": item["category"],
            },
            "vectors": {},
        },
    }
    for item in HOTEL_INFO
]

try:
    data_client.batch_create_data_objects(
        request=(
            vectorsearch_v1beta
            .BatchCreateDataObjectsRequest(
                parent=COLLECTION_PATH,
                requests=batch_request,
            )
        )
    )
    print(f"Created {len(HOTEL_INFO)} data objects")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Data objects already exist (skipping)")
    else:
        raise

### セマンティック検索

自然言語クエリを使用してホテルナレッジベースを検索します。`task_type="QUESTION_ANSWERING"` パラメータにより、エンベディングモデルに質問応答検索用の最適化を指示します。

In [ ]:
import time

# Allow a moment for auto-embeddings to be generated
time.sleep(3)


def vs2_search(query, category="", top_k=3):
    """Search the hotel knowledge base."""
    filter_kwargs = {}
    if category:
        filter_kwargs["filter"] = {
            "category": {"$eq": category}
        }

    request = vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=COLLECTION_PATH,
        semantic_search=vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field="content_embedding",
            task_type="QUESTION_ANSWERING",
            top_k=top_k,
            output_fields=(
                vectorsearch_v1beta.OutputFields(
                    data_fields=["*"],
                )
            ),
            **filter_kwargs,
        ),
    )
    results = search_client.search_data_objects(request)

    parts = [f"### Results for: *{query}*\n"]
    if category:
        parts.append(
            f"**Filter:** category = `{category}`\n\n"
        )
    parts.append("\n")

    count = 0
    for item in results:
        data = item.data_object.data
        title = data.get("title", "")
        cat = data.get("category", "")
        content = data.get("content", "")[:200]
        parts.append(f"**{title}** (`{cat}`)<br>\n")
        parts.append(f"{content}...\n\n")
        count += 1

    if count == 0:
        parts.append("*No results found.*\n")

    display(Markdown("".join(parts)))


vs2_search("What time is checkout?")

In [ ]:
vs2_search("How do I get to the hotel from the airport?")

### フィルター検索

MongoDB スタイルの `$eq` フィルター演算子を使用して、カテゴリで結果を絞り込みます。

In [ ]:
vs2_search("What's available?", category="dining")

In [ ]:
vs2_search("What activities can I do?", category="activities")

---
## 4. 画像検索

`enhancedContent.imageSearch` による画像検索は、Google 検索結果から画像のサムネイルを返します。Concierge AI アプリでは、`web_search` ツールがこの機能を使用して、場所、観光スポット、レストランの写真を表示します。

**注意:** これはプレビュー機能であり、プロジェクトのアローリストへの登録が必要です。プロジェクトで有効になっていない場合、呼び出しは成功しますが画像添付は 0 件となり、通常の Google 検索グラウンディングにフォールバックします。SDK がまだ Agent Platform での `enhancedContent.imageSearch` をサポートしていないため、REST API を使用します。

In [ ]:
payload = json.dumps({
    "contents": [{
        "role": "user",
        "parts": [{
            "text": "Las Vegas Strip at night photos",
        }],
    }],
    "tools": [{
        "googleSearch": {
            "enhancedContent": {"imageSearch": {}},
        },
    }],
}).encode()

resp = get_authorized_session().post(
    rest_endpoint(MODEL_ID),
    data=payload,
    headers={"Content-Type": "application/json"},
    timeout=30,
)
resp.raise_for_status()
data = resp.json()

candidate = data.get("candidates", [{}])[0]
gm = candidate.get("groundingMetadata", {})

parts = ["### Image Search Results\n\n"]
for att in gm.get("attachments", [])[:5]:
    img = att.get("image", {})
    if not img:
        continue
    source = img.get("source", {})
    thumbnail = img.get("thumbnail", {})
    title = source.get("title", "Untitled")
    uri = source.get("uri", "")
    parts.append(f"**{title}**\n")
    parts.append(f"- Source page: {uri}\n")
    thumb_uri = thumbnail.get("uri")
    if thumb_uri:
        parts.append(f"- Thumbnail: ![]({thumb_uri})\n")
    parts.append("\n")

display(Markdown("".join(parts)))

---
## 5. 動画検索（YouTube）

動画検索は、動画寄りのクエリで Google 検索グラウンディングを使用して YouTube コンテンツを検索します。Concierge AI アプリでは、`web_search` ツールがテキスト、画像、マップの検索と並列でこれを実行します。

YouTube の動画 ID は、リダイレクト URL をたどることでグラウンディングチャンクから抽出されます。

In [ ]:
YT_VIDEO_RE = re.compile(
    r"(?:youtube\.com/"
    r"(?:watch\?.*v=|shorts/|embed/|live/|v/)"
    r"|youtu\.be/)"
    r"([A-Za-z0-9_-]{11})"
)


def resolve_youtube_id(redirect_uri):
    """Follow redirect URL, extract YouTube video ID."""
    try:
        resp = http_requests.head(
            redirect_uri, allow_redirects=True, timeout=5,
        )
        m = YT_VIDEO_RE.search(resp.url)
        return m.group(1) if m else ""
    except Exception:
        return ""


payload = json.dumps({
    "contents": [{
        "role": "user",
        "parts": [{
            "text": (
                "Bellagio Fountains Las Vegas "
                "youtube video"
            ),
        }],
    }],
    "tools": [{"googleSearch": {}}],
}).encode()

resp = get_authorized_session().post(
    rest_endpoint(MODEL_ID),
    data=payload,
    headers={"Content-Type": "application/json"},
    timeout=30,
)
resp.raise_for_status()
data = resp.json()

candidate = data.get("candidates", [{}])[0]
gm = candidate.get("groundingMetadata", {})

parts = ["### Video Search Results\n\n"]
found = 0
for chunk in gm.get("groundingChunks", []):
    web = chunk.get("web", {})
    if not web:
        continue
    uri = web.get("uri", "")
    title = web.get("title", "")
    vid = resolve_youtube_id(uri)
    if vid:
        parts.append(f"**{title}**\n")
        parts.append(f"- Video ID: `{vid}`\n")
        yt_url = f"https://www.youtube.com/watch?v={vid}"
        parts.append(f"- URL: {yt_url}\n")
        thumb = (
            "https://i.ytimg.com"
            f"/vi/{vid}/hqdefault.jpg"
        )
        parts.append(f"- Thumbnail: ![]({thumb})\n\n")
        found += 1
    if found >= 3:
        break

if found == 0:
    parts.append(
        "*No YouTube videos found in grounding "
        "chunks. Web results:*\n\n"
    )
    for chunk in gm.get("groundingChunks", [])[:5]:
        web = chunk.get("web", {})
        if web:
            t = web.get("title", "")
            u = web.get("uri", "")
            parts.append(f"- [{t}]({u})\n")

display(Markdown("".join(parts)))

---
## 6. 画像生成のための画像検索

これはセクション 4 のサムネイル画像検索とは異なる機能です。ここでは、`googleSearch.searchTypes.imageSearch` により、画像生成モデルが実際の Google 画像検索結果を画像生成時の**視覚的コンテキスト**として使用できます。

Concierge AI アプリでは、`request_postcard` がこの機能を使用して、実際のウェブ上の写真に基づいた、場所のフォトリアリスティックなセルフィーポストカードを生成します。

**重要:** プロンプトで明示的に検索を促す必要があります（例：「Search Google Images for photos of X, then generate...」）。これがないと、モデルは空の `groundingMetadata` で画像を生成します。

**注意:** これは `gemini-3.1-flash-image-preview`（プレビューモデル）を使用します。

In [ ]:
IMAGE_GEN_MODEL = "gemini-3.1-flash-image-preview"

payload = json.dumps({
    "contents": [{
        "role": "user",
        "parts": [{
            "text": (
                "Search Google Images for photos of "
                "the Bellagio Fountains in Las Vegas, "
                "then use those reference photos to "
                "generate a beautiful watercolor "
                "illustration of the fountains at "
                "sunset with vivid colors."
            ),
        }],
    }],
    "tools": [{
        "googleSearch": {
            "searchTypes": {
                "imageSearch": {},
                "webSearch": {},
            },
        },
    }],
    "generationConfig": {
        "responseModalities": ["TEXT", "IMAGE"],
    },
}).encode()

resp = get_authorized_session().post(
    rest_endpoint(IMAGE_GEN_MODEL),
    data=payload,
    headers={"Content-Type": "application/json"},
    timeout=120,
)
resp.raise_for_status()
data = resp.json()

candidate = data.get("candidates", [{}])[0]

# Display generated image
for part in candidate.get("content", {}).get("parts", []):
    inline = (
        part.get("inlineData")
        or part.get("inline_data")
    )
    if inline and inline.get("data"):
        mime = (
            inline.get("mimeType")
            or inline.get("mime_type")
            or "image/png"
        )
        fmt = mime.split("/")[-1]
        display(IPImage(
            data=base64.b64decode(inline["data"]),
            format=fmt,
        ))
    if "text" in part:
        display(Markdown(part["text"]))

# Display image search sources (attribution)
gm = candidate.get("groundingMetadata", {})
sources = []
for chunk in gm.get("groundingChunks", []):
    img_chunk = chunk.get("image") or {}
    web_chunk = chunk.get("web") or {}
    uri = (
        img_chunk.get("sourceUri")
        or web_chunk.get("uri")
        or ""
    )
    raw_title = (
        img_chunk.get("title")
        or web_chunk.get("title")
        or ""
    )
    title = re.sub(r"</?b>", "", raw_title).strip()
    if uri:
        sources.append(f"- [{title or uri}]({uri})")

if sources:
    display(Markdown(
        "### Image Sources (attribution)\n\n"
        + "\n".join(sources[:5])
    ))
else:
    print(
        "No image search sources returned "
        "(the model may not have triggered "
        "the search)."
    )

---
## クリーンアップ

このノートブックで作成した Vector Search 2.0 コレクションを削除します。

In [ ]:
# Delete the demo collection
try:
    item_ids = [item["id"] for item in HOTEL_INFO]
    reqs = [
        vectorsearch_v1beta.DeleteDataObjectRequest(
            name=(
                f"{COLLECTION_PATH}"
                f"/dataObjects/{obj_id}"
            )
        )
        for obj_id in item_ids
    ]
    data_client.batch_delete_data_objects(
        request=(
            vectorsearch_v1beta
            .BatchDeleteDataObjectsRequest(
                parent=COLLECTION_PATH,
                requests=reqs,
            )
        )
    )
    print(f"Deleted {len(item_ids)} data objects")

    op = vs_client.delete_collection(
        request=(
            vectorsearch_v1beta
            .DeleteCollectionRequest(
                name=COLLECTION_PATH,
            )
        )
    )
    op.result(timeout=300)
    print(f"Collection '{COLLECTION_ID}' deleted")
except Exception as e:
    print(f"Cleanup error: {e}")

継続的な課金を避けるために：

1. 不要になった場合は、Google Cloud プロジェクトを削除してください
2. または [Agent Platform](https://console.cloud.google.com/apis/api/aiplatform.googleapis.com) と [Vector Search](https://console.cloud.google.com/apis/api/vectorsearch.googleapis.com) API を無効にしてください